# Exploratory Data Analysis (EDA) - Events

Este notebook corrige el EDA de eventos para alinearlo con el taller. En vez de depender de `events.csv`, trabajamos con `shots_with_qualifiers.csv`, que si conserva la columna `qualifiers` y permite analizar la "mina de oro" del proyecto: `BigChance`, `Penalty`, parte del cuerpo y zona del disparo.


In [ ]:
import os

# --- SOLUCION DE RUTAS (CWD) 100% PORTABLE ---
if os.path.exists('CSV (API)') and os.path.exists('EDA (Analisis Exploratorio de Datos)'):
    os.chdir('EDA (Analisis Exploratorio de Datos)')
elif os.path.exists('CSV (API)') and os.path.exists('EDA (Análisis Exploratorio de Datos)'):
    os.chdir('EDA (Análisis Exploratorio de Datos)')

print('Directorio configurado en:', os.getcwd())


In [ ]:
import ast
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.patches import Arc, Rectangle

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

df_shots = pd.read_csv('../CSV (API)/shots_with_qualifiers.csv')
df_shots = df_shots[(df_shots['x'] != 0) | (df_shots['y'] != 0)].copy()

print(f'Tiros cargados para EDA: {df_shots.shape[0]}')
df_shots.head(3)


## 1. Limpieza y Feature Engineering

Seguimos la guia del taller: nos quedamos solo con tiros validos, parseamos `qualifiers` y construimos variables derivadas geometricas y contextuales.


In [ ]:
def parse_qualifiers(value):
    if not isinstance(value, str) or not value.strip():
        return []
    try:
        return ast.literal_eval(value)
    except (ValueError, SyntaxError):
        return []

def get_zone_bucket(qualifiers):
    detailed_zones = {
        'BoxCentre', 'BoxLeft', 'BoxRight',
        'OutOfBoxCentre', 'OutOfBoxLeft', 'OutOfBoxRight',
        'SmallBoxCentre', 'SmallBoxLeft', 'SmallBoxRight'
    }
    for item in qualifiers:
        display = item.get('type', {}).get('displayName')
        if display in detailed_zones:
            return display
    for item in qualifiers:
        if item.get('type', {}).get('displayName') == 'Zone':
            return item.get('value', 'Unknown')
    return 'Unknown'

q = df_shots['qualifiers'].fillna('')
df_shots['qualifiers_parsed'] = df_shots['qualifiers'].apply(parse_qualifiers)
df_shots['is_big_chance'] = q.str.contains('BigChance', na=False).astype(int)
df_shots['is_penalty'] = q.str.contains('Penalty', na=False).astype(int)
df_shots['is_header'] = q.str.contains('Head', na=False).astype(int)
df_shots['is_right_foot'] = q.str.contains('RightFoot', na=False).astype(int)
df_shots['is_left_foot'] = q.str.contains('LeftFoot', na=False).astype(int)

df_shots['distance_to_goal'] = np.sqrt((100 - df_shots['x'])**2 + (50 - df_shots['y'])**2)
df_shots['shot_angle'] = np.abs(np.arctan2(50 - df_shots['y'], 100 - df_shots['x']))
df_shots['shot_angle_degrees'] = np.degrees(df_shots['shot_angle'])
df_shots['shot_zone'] = df_shots['qualifiers_parsed'].apply(get_zone_bucket)

df_shots['body_part'] = np.select(
    [df_shots['is_header'] == 1, df_shots['is_right_foot'] == 1, df_shots['is_left_foot'] == 1],
    ['Head', 'RightFoot', 'LeftFoot'],
    default='Other'
)

df_shots[['player_name', 'is_goal', 'is_big_chance', 'is_penalty', 'body_part', 'shot_zone', 'distance_to_goal', 'shot_angle_degrees']].head()


## 2. Visualizacion 1: Shot Map sobre cancha de futbol

Objetivo: confirmar espacialmente la relacion entre la ubicacion del disparo y la conversion a gol usando una cancha de futbol como contexto visual.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
ax.set_facecolor('#2f6f3e')

# Mitad ofensiva de la cancha
ax.add_patch(Rectangle((50, 0), 50, 100, fill=False, edgecolor='white', linewidth=2))
ax.plot([50, 50], [0, 100], color='white', linewidth=2)
ax.add_patch(Arc((50, 50), 18.3, 18.3, angle=0, theta1=270, theta2=90, color='white', linewidth=2))
ax.add_patch(Rectangle((83, 21.1), 17, 57.8, fill=False, edgecolor='white', linewidth=2))
ax.add_patch(Rectangle((94.2, 36.8), 5.8, 26.4, fill=False, edgecolor='white', linewidth=2))
ax.scatter(88.5, 50, color='white', s=18)
ax.add_patch(Arc((88.5, 50), 18.3, 18.3, angle=0, theta1=128, theta2=232, color='white', linewidth=2))
ax.plot([100, 100], [45.2, 54.8], color='white', linewidth=4)

palette = {0: '#7ec8e3', 1: '#d62728'}
labels = {0: 'No Gol', 1: 'Gol'}
for goal_value in [0, 1]:
    subset = df_shots[df_shots['is_goal'] == goal_value]
    ax.scatter(
        subset['x'],
        subset['y'],
        c=palette[goal_value],
        s=28 if goal_value == 0 else 40,
        alpha=0.65 if goal_value == 0 else 0.85,
        edgecolors='black',
        linewidths=0.25,
        label=labels[goal_value]
    )

ax.set_xlim(50, 101)
ax.set_ylim(0, 100)
ax.set_title('Mapa de tiros sobre cancha de futbol')
ax.set_xlabel('Coordenada X')
ax.set_ylabel('Coordenada Y')
ax.legend(loc='upper left')
plt.show()


**Insight 1:** Al dibujar los tiros sobre una cancha de futbol, se aprecia mucho mejor que los goles se concentran cerca del arco rival y en el carril central. Los remates lejanos o muy abiertos aparecen con frecuencia, pero rara vez terminan en gol.


## 3. Visualizacion 2: Volumen de Tiros vs Distancia al Arco

Objetivo: comparar el volumen de tiros por distancia euclidiana a la porteria, usando `is_goal` como variable binaria.


In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(
    data=df_shots,
    x='distance_to_goal',
    hue='is_goal',
    multiple='stack',
    bins=30,
    palette={0: '#6baed6', 1: '#d62728'}
)
plt.title('Volumen de Tiros vs Distancia al Arco')
plt.xlabel('Distancia Euclidiana a la porteria')
plt.ylabel('Cantidad de Tiros')
plt.show()


**Insight 2:** La mayor parte de los tiros se concentra en distancias medias, pero los goles se apilan sobre todo en los rangos mas cercanos al arco. A medida que la distancia aumenta, el volumen total puede seguir siendo alto, pero la proporcion de tiros que terminan en gol cae con fuerza.


## 4. Visualizacion 3: Densidad de Tiros segun el Angulo

Objetivo: comparar la densidad de tiros por angulo de tiro absoluto, usando `is_goal` como variable binaria.


In [ ]:
plt.figure(figsize=(10, 6))
sns.kdeplot(
    data=df_shots,
    x='shot_angle_degrees',
    hue='is_goal',
    common_norm=False,
    fill=True,
    palette={0: '#6baed6', 1: '#d62728'}
)
plt.title('Densidad de Tiros segun el Angulo')
plt.xlabel('Angulo de tiro (absoluto)')
plt.ylabel('Densidad')
plt.show()


**Insight 3:** La distribucion de goles se concentra en angulos mas favorables, mientras que los no goles dominan buena parte del resto de la distribucion. Eso confirma que el angulo del disparo ayuda a separar tiros peligrosos de remates con poca probabilidad de conversion.
